## Transformaciones: planteamiento

Qué transformaciones se hacen, en qué orden y para qué técnica, antes de ejecutarlas en `b_transformaciones` y `c_uso`

- **Corre en:** no ejecuta código (solo texto)

---
### 1. Objetivo

- Dejar lista, **sobre Spark**, la base relacional y los artefactos que alimentan las Partes II–VI
- Partir del staging del EDA (`00_EDA/a_entendimiento`) y de los problemas de calidad ya identificados (§3, "Acciones
  previstas")

---
### 2. Grano de análisis

- **Una fila = una adjudicación de una entidad a un proveedor en una familia UNSPSC** (entidad–proveedor–familia)
- **Por qué este grano:**
  - Las técnicas trabajan sobre **relaciones**, no sobre procesos aislados: el grano del P1 (un proceso) no tenía al
    proveedor como actor
  - La **adjudicación** es la relación efectiva; el contrato falta en el 10.8% de los casos
  - La **familia** (4 dígitos UNSPSC) define el mercado: el segmento (2 dígitos) es muy grueso y el producto (8 dígitos)
    muy disperso
- Todo lo demás (features, matrices, grafos) se agrega desde este grano

---
### 3. Capas de datos

| Capa | Contenido | Grano | Partición | Notebook |
|---|---|---|---|---|
| `data/raw/` | JSON OCDS mensual | archivo | año / mes | `data/download.py` |
| `data/bronze/` | OCDS anidado en Parquet, sin `documents` | proceso | — | `00_EDA/a_entendimiento` |
| `data/staging/` | 6 tablas aplanadas (procesos, adjudicaciones, oferentes, ítems, contratos, partes) | según tabla | `anio` | `00_EDA/a_entendimiento` |
| `data/final/` | procesos limpios, hechos, participantes, dimensiones, auditoría de la resolución | entidad–proveedor–familia | `anio/mes` | `b_transformaciones` |
| `data/artefactos/` | features, matrices, índices y grafos | según artefacto | — | `c_uso` |

---
### 4. Transformaciones

#### a. Limpieza (`b_transformaciones` §1)

- Texto en mayúsculas y espacios colapsados; 1 fila por `ocid`
- Solo procesos con fecha en 2023-2025
- Montos a soles con el tipo de cambio mensual del BCRP; monto ≤ 0 → nulo
- Banderas de competencia: `postor_unico` (n_postores = 1) y `es_directa` (contratación directa)
- **No** se filtra por estado del award: la fuente no lo publica

#### b. Resolución de entidades (`b_transformaciones` §2)

| Nivel | Regla | Resultado |
|---|---|---|
| 1. Identificador | `PE-CONSUCODE-…` → `ENT:`; RUC → `RUC:`; RUC 10 / DNI → `PN:`; código de consorcio → `CONS:`; extranjero → `EXT:` | Determinista |
| 2. Nombre exacto | Mención sin id con el mismo nombre normalizado que **una sola** clave → esa clave | Determinista |
| 3. MinHash + LSH | Mención sin id → clave más parecida si Jaccard ≥ 0.85 (sin consorcios); si no, `NOM:` | Probabilístico |

- Claves distintas nunca se fusionan; los casi homónimos (Jaccard ≥ 0.95) se guardan como **posibles vinculadas**
- Las claves `PN:` contienen el DNI de personas naturales: se usan internamente; en la web solo se publican agregados

#### c. Hechos y participantes (`b_transformaciones` §3)

- **Hechos:** adjudicación × proveedor resuelto × familia, con `monto_asignado` = monto / ganadores / familias (la suma
  reproduce el monto adjudicado)
- **Participantes:** oferentes ∪ ganadores por proceso, deduplicados por clave resuelta

---
### 5. Artefactos por técnica (`c_uso`)

| Técnica (Parte) | Transformación | Artefacto |
|---|---|---|
| Clustering y anomalías (II) | Agregar hechos por actor: tamaño, alcance, dependencia (HHI), competencia, tipo | `features_proveedor`, `features_entidad` |
| Recomendación (III) | Pivotear hechos a entidad × proveedor y entidad × familia, por año, con índices enteros | `matriz_entidad_proveedor`, `matriz_entidad_familia`, `indice_*` |
| PageRank y HITS (IV) | Aristas dirigidas entidad → proveedor con peso | `grafo_bipartito` |
| Comunidades (V) | Self-join de participantes por proceso → pares de proveedores | `grafo_coparticipacion` |
| Representación (VI) | Reutiliza la matriz (SVD) y la co-participación (Node2Vec) | — |

---
### 6. Criterio de aristas

- **Bipartito:** entidad → proveedor si hubo ≥ 1 adjudicación; peso = n.º de adjudicaciones o monto
- **Co-participación:** proveedor — proveedor si participaron juntos en **≥ 2 procesos** de **2 a 30 participantes**;
  peso Σ 1/(n−1)
  - El tope de 30 evita que los procesos masivos (n(n−1)/2 pares) dominen con vínculos débiles
  - El umbral de 2 descarta coincidencias únicas
- **Alternativa:** grafo KNN sobre `features_proveedor` estandarizado, si la co-participación resultara escasa
- **Consorcios:** son un nodo propio; la fuente no publica sus miembros, así que no hay arista consorcio–empresa

---
### 7. Vínculos incompletos

| Caso | Tratamiento |
|---|---|
| Adjudicación sin contrato (10.8%) | Se conserva como arista; `tiene_contrato = 0` |
| Contrato sin adjudicación (1.0%) | No genera arista: no tiene proveedor identificable |
| Award sin proveedor (1.9%) | No genera arista |
| Ganador no listado como oferente | Se agrega a participantes (`en_lista_oferentes = 0`, `gano = 1`) |
| Ítem sin UNSPSC | Familia del tender; si no hay, `SIN` (excluida de las matrices de rubro) |

---
### 8. Particionamiento

- **Disco:** `hechos`, `procesos` y `participantes` por `anio/mes`. Los filtros temporales (hold-out 2025) leen solo esas
  carpetas (*partition pruning*)
- **Self-join de co-participación:** se reparticiona una vez por `ocid` y se cachea, para que ambos lados del join ya estén
  co-particionados y no haya otro shuffle
- **Dimensiones chicas** (entidades, mapa de la resolución, tipo de cambio): *broadcast join*, sin mover la tabla grande
- **`spark.sql.shuffle.partitions` = 24:** con 2 workers de 1 core, más particiones solo agregan overhead
- El efecto en el plan físico (`explain()`) y en el tiempo se mide en `c_uso` §4